# 3. Multivariate Gaussian  [3.6]

TOML-MIRI — Linear Algebra for Data Science, exploration exercises.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

np.set_printoptions(precision=4, suppress=True)

def check(label, ok):
    print(f"{'✓' if ok else '✗'}  {label}")

def rot(theta_deg):
    t = np.deg2rad(theta_deg)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

def sorted_eigh(S):
    # eigen-decomposition of a symmetric matrix, eigenvalues in DEcreasing order
    lam, V = np.linalg.eigh(S)
    idx = np.argsort(lam)[::-1]
    return lam[idx], V[:, idx]

---
## 3.1 Build $\Sigma$ from geometry

**Task.** Build a $2\times2$ covariance $\Sigma = Q\Lambda Q^\top$ from a rotation angle $\theta$ and two variances $\lambda_1, \lambda_2$ (sliders),

$$Q = \begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}, \qquad \Lambda = \begin{bmatrix}\lambda_1 & 0\\ 0 & \lambda_2\end{bmatrix}.$$

Sample $N$ points from $\mathcal{N}(\mu, \Sigma)$, plot them with the ellipses $(x-\mu)^\top\Sigma^{-1}(x-\mu) = c^2$ for $c = 1, 2, 3$, and draw the eigenvectors scaled by $c\sqrt{\lambda_i}$.

We use $\mu = [5, 5]^\top$ as in the figure after 3.6.4.

### Prediction (write this *before* running)

- Roughly what fraction of the points falls inside the $c=1$, $c=2$, $c=3$ ellipses?
- Which way does the cloud point for $\theta = 45°$, $\lambda_1 > \lambda_2$?

*(fill in)*

In [ ]:
mu = np.array([5.0, 5.0])

def make_sigma(theta_deg, lam1, lam2):
    Q = rot(theta_deg)
    return Q @ np.diag([lam1, lam2]) @ Q.T, Q

def ellipse(mu, Sigma, c, n=200):
    # points x with (x − μ)ᵀ Σ⁻¹ (x − μ) = c²  :  x = μ + Q (c √λ ∘ [cos φ, sin φ])
    lam, Q = np.linalg.eigh(Sigma)
    phi = np.linspace(0, 2 * np.pi, n)
    return mu[:, None] + Q @ (c * np.sqrt(lam)[:, None] * np.vstack([np.cos(phi), np.sin(phi)]))

def mahalanobis2(X, mu, Sigma):
    # (x − μ)ᵀ Σ⁻¹ (x − μ) for each row x of X
    D = X - mu
    return np.einsum("ij,jk,ik->i", D, np.linalg.inv(Sigma), D)

def plot_gaussian(theta=45, lam1=2.0, lam2=0.5, N=1000, seed=0, ax=None):
    Sigma, Q = make_sigma(theta, lam1, lam2)
    X = np.random.default_rng(seed).multivariate_normal(mu, Sigma, N)
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.scatter(*X.T, s=3, alpha=min(1.0, 300 / N), color="tab:blue")
    ax.plot([0, mu[0]], [0, mu[1]], color="red", lw=1)               # the vector μ, as in the notes
    d2 = mahalanobis2(X, mu, Sigma)
    rows = []
    for c, col in zip([1, 2, 3], ["gold", "darkorange", "crimson"]):
        ax.plot(*ellipse(mu, Sigma, c), color=col, lw=2, label=f"c = {c}")
        rows.append((c, np.mean(d2 <= c**2), chi2.cdf(c**2, df=2)))
    for i, lam_i in enumerate([lam1, lam2]):
        q = Q[:, i]
        ax.plot(*np.column_stack([mu, mu + 3 * np.sqrt(lam_i) * q]), "k-", lw=2)
        ticks = mu[:, None] + np.outer(q, [1, 2, 3]) * np.sqrt(lam_i)    # c √λᵢ qᵢ, c = 1,2,3
        ax.plot(*ticks, "ko", ms=4)
        ax.text(*(mu + 3.3 * np.sqrt(lam_i) * q), f"q{i+1}", fontsize=11)
    L = max(5.0, 3.5 * np.sqrt(max(lam1, lam2)))
    ax.set_xlim(mu[0] - L, mu[0] + L); ax.set_ylim(mu[1] - L, mu[1] + L)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_title(f"N = {N}")
    if own_fig:
        ax.legend(loc="upper left")
        plt.show()
    return rows

def print_fractions(rows):
    print(" c | fraction inside | χ²(2) CDF at c² = 1 − exp(−c²/2)")
    for c, frac, p in rows:
        print(f" {c} |     {frac:.4f}      |   {p:.4f}")

In [ ]:
print_fractions(plot_gaussian(theta=45, lam1=2.0, lam2=0.5, N=1000))

### Verify numerically

In [ ]:
Sigma, Q = make_sigma(30, 2.0, 0.5)
check("Σ is symmetric", np.allclose(Sigma, Sigma.T))
check("Σ qᵢ = λᵢ qᵢ", np.allclose(Sigma @ Q, Q @ np.diag([2.0, 0.5])))
for c in [1, 2, 3]:
    check(f"ellipse points satisfy (x−μ)ᵀΣ⁻¹(x−μ) = {c}²",
          np.allclose(mahalanobis2(ellipse(mu, Sigma, c).T, mu, Sigma), c**2))
cs = np.array([1, 2, 3])
check("χ² CDF with 2 dof at c² equals 1 − exp(−c²/2)", np.allclose(chi2.cdf(cs**2, 2), 1 - np.exp(-cs**2 / 2)))

### Explore: N = 100, 1,000, 100,000 (as in the figure after 3.6.4). Set $\lambda_1 = \lambda_2$: what happens to the ellipses?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, N in zip(axes, [100, 1000, 100_000]):
    rows = plot_gaussian(theta=-45, lam1=2.0, lam2=1.0, N=N, ax=ax)
    print(f"N = {N}"); print_fractions(rows); print()
plt.show()

In [ ]:
@interact(theta=FloatSlider(min=0, max=180, step=5, value=45),
          lam1=FloatSlider(min=0.1, max=4, step=0.1, value=2.0),
          lam2=FloatSlider(min=0.1, max=4, step=0.1, value=0.5),
          N=Dropdown(options=[100, 1000, 10_000, 100_000], value=1000),
          seed=IntSlider(min=0, max=20, value=0))
def explore_gaussian(theta, lam1, lam2, N, seed):
    print_fractions(plot_gaussian(theta, lam1, lam2, N, seed))

---
### Explain (hand-in — in your own words)

**1. What fraction of points falls inside each ellipse? Compare with the $\chi^2$ distribution with 2 degrees of freedom.**
*(Hint: if $x = \mu + Q\Lambda^{1/2}z$ with $z \sim \mathcal{N}(0, I)$, what is $(x-\mu)^\top\Sigma^{-1}(x-\mu)$ in terms of $z$?)*

*Your answer.*

**2. What happens to the ellipses when $\lambda_1 = \lambda_2$? Does $\theta$ still matter?**

*Your answer.*

---
## 3.2 Generate correlated data from white noise

**Task.** Sample $z \sim \mathcal{N}(0, I)$, apply the transformation below, and compare with `np.random.multivariate_normal`.

$$x = \mu + Q\Lambda^{1/2} z \quad\Longrightarrow\quad \mathrm{Cov}(x) = Q\Lambda^{1/2} I \Lambda^{1/2} Q^\top = \Sigma.$$

### Prediction (write this *before* running)

- Shape of the cloud after each step: $z$ → $\Lambda^{1/2}z$ → $Q\Lambda^{1/2}z$ → $+\mu$.
- What would $\mathrm{Cov}(x)$ be if you used $\Lambda$ instead of $\Lambda^{1/2}$?

*(fill in)*

In [ ]:
theta, lam1, lam2 = 30, 3.0, 0.5
Sigma, Q = make_sigma(theta, lam1, lam2)
Lam_half = np.diag(np.sqrt([lam1, lam2]))

N = 20_000
rng = np.random.default_rng(1)
Z = rng.standard_normal((2, N))                     # white noise, one sample per column
X = mu[:, None] + Q @ Lam_half @ Z                  # rotate–stretch (and shift)
X_np = rng.multivariate_normal(mu, Sigma, N).T      # NumPy's sampler
X_wrong = mu[:, None] + Q @ np.diag([lam1, lam2]) @ Z

print("true Σ:\n", Sigma)
print("\nsample Cov(μ + QΛ^{1/2} z):\n", np.cov(X))
print("\nsample Cov(np.random.multivariate_normal):\n", np.cov(X_np))
print("\nsample Cov using Λ instead of Λ^{1/2}:\n", np.cov(X_wrong))
print("QΛ²Qᵀ:\n", Q @ np.diag([lam1**2, lam2**2]) @ Q.T)

In [ ]:
k = 2000                                            # points to draw per panel
ang = np.arctan2(Z[1, :k], Z[0, :k])                # colour = angle of the original z
stages = [Z[:, :k], Lam_half @ Z[:, :k], Q @ Lam_half @ Z[:, :k], X[:, :k], X_np[:, :k]]
titles = ["z ~ N(0, I)", "Λ^{1/2} z  (stretch)", "Q Λ^{1/2} z  (rotate)", "μ + Q Λ^{1/2} z", "np.random.multivariate_normal"]
fig, axes = plt.subplots(1, 5, figsize=(22, 4.6))
for i, (ax, P, title) in enumerate(zip(axes, stages, titles)):
    if i < 4:
        ax.scatter(*P, c=ang, cmap="hsv", s=3)
    else:
        ax.scatter(*P, s=3, color="gray")
    c0 = mu if i >= 3 else np.zeros(2)
    ax.plot(*ellipse(c0, Sigma if i >= 2 else (Lam_half**2 if i == 1 else np.eye(2)), 2), "k", lw=1)
    ax.set_xlim(c0[0] - 6, c0[0] + 6); ax.set_ylim(c0[1] - 6, c0[1] + 6)
    ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title(title)
plt.show()

### Verify numerically

In [ ]:
M = Q @ Lam_half
check("(QΛ^{1/2})(QΛ^{1/2})ᵀ = Σ exactly", np.allclose(M @ M.T, Sigma))
check("sample Cov of μ + QΛ^{1/2}z ≈ Σ", np.allclose(np.cov(X), Sigma, atol=0.1))
check("sample Cov of multivariate_normal ≈ Σ", np.allclose(np.cov(X_np), Sigma, atol=0.1))
check("sample mean ≈ μ", np.allclose(X.mean(axis=1), mu, atol=0.05))
check("using Λ gives Cov ≈ QΛ²Qᵀ (≠ Σ)", np.allclose(np.cov(X_wrong), Q @ np.diag([lam1**2, lam2**2]) @ Q.T, atol=0.4))
L = np.linalg.cholesky(Sigma)
check("Cholesky factor L also satisfies L Lᵀ = Σ (another valid 'square root')", np.allclose(L @ L.T, Sigma))

---
### Explain (hand-in — in your own words)

**1. Read the transformation as rotate–stretch–rotate [3.1]. Which steps are present here and what does each one do to the white-noise cloud?**
*(Hint: $z$ is already rotation-invariant, so the first rotation $Q^\top$ of $S = Q\Lambda Q^\top$ is not needed.)*

*Your answer.*

**2. Why is $\Lambda^{1/2}$ (and not $\Lambda$) the right stretch?**
*(Hint: $\mathrm{Cov}(Mz) = M\,\mathrm{Cov}(z)\,M^\top$. Variance scales with the square of the stretch.)*

*Your answer.*

---
## 3.3 Recover the parameters

**Task.** Estimate $\mu$ and the sample covariance from the samples, then compare its eigenvectors and eigenvalues with the true ones as $N$ grows.

$$\bar{x} = \frac{1}{N}\sum_{i=1}^N x_i, \qquad S = \frac{1}{N-1}\sum_{i=1}^N (x_i - \bar{x})(x_i - \bar{x})^\top.$$

Error measures (averaged over 50 repetitions for each $N$):
- direction error of eigenvector $i$: the angle between $\hat v_i$ and $v_i$ (sign ignored)
- relative eigenvalue error $|\hat\lambda_i - \lambda_i| / \lambda_i$
- mean error $\|\bar{x} - \mu\|$

### Prediction (write this *before* running)

- How fast do the errors shrink with $N$ (which power of $N$)?
- Which is estimated worse: the direction of the long axis or of the short axis?
- What happens to the direction error when $\lambda_1 \approx \lambda_2$?

*(fill in)*

In [ ]:
def sample_stats(X):
    # X: N × d, one sample per row
    N = X.shape[0]
    xbar = X.mean(axis=0)
    D = X - xbar
    return xbar, D.T @ D / (N - 1)

def estimation_errors(Sigma, mu, Ns, trials=50, seed=0):
    rng = np.random.default_rng(seed)
    lam_true, V_true = sorted_eigh(Sigma)
    d = len(mu)
    ang, rel, merr = np.zeros((len(Ns), d)), np.zeros((len(Ns), d)), np.zeros(len(Ns))
    for a, N in enumerate(Ns):
        for _ in range(trials):
            xbar, S = sample_stats(rng.multivariate_normal(mu, Sigma, N))
            lam, V = sorted_eigh(S)
            cos = np.clip(np.abs(np.sum(V * V_true, axis=0)), 0, 1)
            ang[a] += np.degrees(np.arccos(cos)) / trials
            rel[a] += np.abs(lam - lam_true) / lam_true / trials
            merr[a] += np.linalg.norm(xbar - mu) / trials
    return ang, rel, merr

def plot_errors(Sigma, mu, title, Ns=np.unique(np.logspace(1, 4.5, 12).astype(int))):
    ang, rel, merr = estimation_errors(Sigma, mu, Ns)
    lam = sorted_eigh(Sigma)[0]
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    for i in range(len(lam)):
        axes[0].loglog(Ns, ang[:, i], "o-", label=f"v{i+1} (λ = {lam[i]:g})")
        axes[1].loglog(Ns, rel[:, i], "o-", label=f"λ{i+1} = {lam[i]:g}")
    axes[2].loglog(Ns, merr, "o-", label="‖x̄ − μ‖")
    for ax, ylabel in zip(axes, ["angle error (degrees)", "relative eigenvalue error", "mean error"]):
        ref = ax.get_lines()[0].get_ydata()[0] * np.sqrt(Ns[0] / Ns)
        ax.loglog(Ns, ref, "k--", lw=1, label="∝ N^(−1/2)")
        ax.set_xlabel("N"); ax.set_ylabel(ylabel); ax.grid(alpha=0.3, which="both"); ax.legend()
    fig.suptitle(title)
    plt.show()

### Verify the estimator against NumPy

In [ ]:
Xs = np.random.default_rng(0).multivariate_normal(mu, Sigma, 500)
xbar, S = sample_stats(Xs)
check("x̄ = np.mean", np.allclose(xbar, Xs.mean(axis=0)))
check("S = np.cov (which uses N − 1)", np.allclose(S, np.cov(Xs.T)))
check("S is symmetric PSD", np.allclose(S, S.T) and np.linalg.eigvalsh(S).min() >= 0)

In [ ]:
Sigma_a, _ = make_sigma(30, 3.0, 0.5)
plot_errors(Sigma_a, mu, "well-separated eigenvalues: λ = (3, 0.5)")

What it looks like: true (black) vs estimated (colour) $c = 2$ ellipses and axes for small $N$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
rng = np.random.default_rng(3)
for ax, N in zip(axes, [10, 100, 1000]):
    ax.plot(*ellipse(mu, Sigma_a, 2), "k", lw=2.5, label="true")
    for rep in range(5):
        Xs = rng.multivariate_normal(mu, Sigma_a, N)
        xbar, S = sample_stats(Xs)
        ax.plot(*ellipse(xbar, S, 2), lw=1, alpha=0.8)
    ax.set_aspect("equal"); ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.grid(alpha=0.3)
    ax.set_title(f"5 estimates with N = {N}")
plt.show()

### Explore: $\lambda_1 \approx \lambda_2$

In [ ]:
Sigma_b, _ = make_sigma(30, 1.0, 0.9)
plot_errors(Sigma_b, mu, "nearly equal eigenvalues: λ = (1, 0.9)")

### Explore further: 3-D

In 2-D the two eigenvectors are forced to be perpendicular, so their direction errors are always the same. In 3-D they can differ. Here $\lambda = (4, 1, 0.8)$: one eigenvalue is well separated, two are close.

In [ ]:
Q3, _ = np.linalg.qr(np.random.default_rng(7).standard_normal((3, 3)))
Sigma3 = Q3 @ np.diag([4.0, 1.0, 0.8]) @ Q3.T
plot_errors(Sigma3, np.zeros(3), "3-D: λ = (4, 1, 0.8)")

---
### Explain (hand-in — in your own words)

**1. Which is estimated worse, the direction of the long axis or the short axis?**
*(Hint: compare the 2-D and 3-D plots. What does a direction error depend on: the size of its own eigenvalue or the gap to the others?)*

*Your answer.*

**2. What happens when $\lambda_1 \approx \lambda_2$? Why is the direction then hard (or even meaningless) to estimate?**
*(Hint: think of the case $\lambda_1 = \lambda_2$ exactly: which directions are eigenvectors?)*

*Your answer.*

---
## 3.4 The precision matrix

**Task.** Using $\Sigma$ from Example 3.3, plot the paraboloid (Figure 6) and the density surface side by side:

$$z = \tfrac12 (x-\mu)^\top\Sigma^{-1}(x-\mu), \qquad f_X(x) = \frac{1}{(2\pi)^{n/2}|\Sigma|^{1/2}}\, e^{-z}.$$

Example 3.3: $\Sigma = \begin{bmatrix} 3/2 & -1/2 \\ -1/2 & 3/2\end{bmatrix}$, $\mu = [5, 5]^\top$.

### Prediction (write this *before* running)

- Eigenvalues of $\Sigma$: ___; of $\Sigma^{-1}$: ___.
- Along which direction is the paraboloid flattest?

*(fill in)*

In [ ]:
Sigma = np.array([[1.5, -0.5], [-0.5, 1.5]])
mu = np.array([5.0, 5.0])
P = np.linalg.inv(Sigma)                         # precision matrix

def paraboloid_and_density(X1, X2):
    D1, D2 = X1 - mu[0], X2 - mu[1]
    Z = 0.5 * (P[0, 0] * D1**2 + 2 * P[0, 1] * D1 * D2 + P[1, 1] * D2**2)
    f = np.exp(-Z) / (2 * np.pi * np.sqrt(np.linalg.det(Sigma)))
    return Z, f

g = np.linspace(0, 10, 101)
X1, X2 = np.meshgrid(g, g)
Z, f = paraboloid_and_density(X1, X2)

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                    subplot_titles=["paraboloid  ½ (x−μ)ᵀ Σ⁻¹ (x−μ)", "density  f_X(x)"])
fig.add_trace(go.Surface(x=X1, y=X2, z=Z, colorscale="Viridis", showscale=False), row=1, col=1)
fig.add_trace(go.Surface(x=X1, y=X2, z=f, colorscale="Viridis", showscale=False), row=1, col=2)
fig.update_layout(height=550)
fig.show()

In [ ]:
lam, V = sorted_eigh(Sigma)
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, F, title in [(axes[0], Z, "contours of z"), (axes[1], f, "contours of f_X")]:
    cs = ax.contour(X1, X2, F, levels=12, cmap="viridis")
    ax.clabel(cs, fontsize=7)
    for i, col in enumerate(["crimson", "royalblue"]):
        ax.annotate("", xy=mu + 2 * np.sqrt(lam[i]) * V[:, i], xytext=mu,
                    arrowprops=dict(arrowstyle="->", color=col, lw=2.5))
        ax.text(*(mu + 2.4 * np.sqrt(lam[i]) * V[:, i]), f"λ(Σ)={lam[i]:g}\nλ(Σ⁻¹)={1/lam[i]:g}",
                color=col, ha="center")
    ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title(title)
plt.show()

### Verify numerically

In [ ]:
lamS, QS = sorted_eigh(Sigma)
lamP, QP = sorted_eigh(P)
Q_notes = np.array([[np.sqrt(2) / 2, -np.sqrt(2) / 2], [np.sqrt(2) / 2, np.sqrt(2) / 2]])

check("eigenvalues of Σ are {2, 1}", np.allclose(lamS, [2, 1]))
check("Σ = Q diag(1, 2) Qᵀ with Q from the notes", np.allclose(Sigma, Q_notes @ np.diag([1, 2]) @ Q_notes.T))
check("Σ⁻¹ = [[3/4, 1/4], [1/4, 3/4]] (notes)", np.allclose(P, [[0.75, 0.25], [0.25, 0.75]]))
check("eigenvalues of Σ⁻¹ are 1/λ(Σ)", np.allclose(np.sort(lamP), np.sort(1 / lamS)))
check("same eigenvectors: Σ⁻¹ qᵢ = (1/λᵢ) qᵢ", np.allclose(P @ QS, QS / lamS))
check("|Σ| = λ₁ λ₂ = 2  [3.6.6]", np.isclose(np.linalg.det(Sigma), np.prod(lamS)) and np.isclose(np.prod(lamS), 2))
check("z = 0 only at μ (minimum of the paraboloid)", np.isclose(Z.min(), 0) and np.allclose(
      [X1.flat[Z.argmin()], X2.flat[Z.argmin()]], mu))
gg = np.linspace(-10, 20, 601)
G1, G2 = np.meshgrid(gg, gg)
_, fbig = paraboloid_and_density(G1, G2)
check("density integrates to 1", np.isclose(fbig.sum() * (gg[1] - gg[0]) ** 2, 1, atol=1e-4))

---
### Explain (hand-in — in your own words)

**1. How are the eigenvalues of $\Sigma$ and $\Sigma^{-1}$ related?**
*(Hint: start from $\Sigma = Q\Lambda Q^\top$ and invert each factor.)*

*Your answer.*

**2. Why does the long axis of the ellipse correspond to the *small* eigenvalue of $\Sigma^{-1}$?**
*(Hint: along an eigenvector $q_i$, $z = \tfrac{1}{2}\, t^2 / \lambda_i$. How far do you have to go to reach $z = 1$?)*

*Your answer.*

**3. Check $|\Sigma| = \lambda_1\lambda_2$ [3.6.6]. What does $|\Sigma|^{1/2}$ do in the density formula?**
*(Hint: it is the area scaling of the map $z \mapsto Q\Lambda^{1/2}z$.)*

*Your answer.*